<a href="https://colab.research.google.com/github/kantathamangjam-lgtm/GenAI/blob/main/Study_Buddy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf2 openai
import os
from openai import OpenAI
import PyPDF2

from google.colab import userdata
# Securely retrieve the key from Colab Secrets
#os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Initialize the client (it automatically reads the environment variable)
#client = OpenAI()

# 1. Initialize the OpenAI client (ensure OPENAI_API_KEY is set in your environment)
client = OpenAI(api_key="")


def extract_text_from_pdf(pdf_path: str) -> str:
  """Extracts all text from a given PDF file."""
  text = ""
  try:
    with open(pdf_path, "rb") as file:
      reader = PyPDF2.PdfReader(file)
      print(f"Total pages in PDF: {len(reader.pages)}")
      for page_num, page in enumerate(reader.pages):
        extracted = page.extract_text()
        if extracted:
          text += f"\n--- Page {page_num + 1} ---\n" + extracted
  except Exception as e:
    print(f"Error reading PDF: {e}")
  return text


def summarize_text_with_agent(text: str) -> str:
  """Sends the extracted PDF text to an AI agent for a structured summary."""
  prompt = f"""
    You are an expert AI analysis agent.
    Read the following document text and provide a clear, structured summary containing:
    1. Main Theme / Core Objective
    2. Key Takeaways (bullet points)
    3. Important Data or Findings
    4. Actionable Conclusions / Recommendations

    Document Text:
    {text[:15000]}  # Truncate or chunk if the text exceeds token limits
    """

  response = client.chat.completions.create(
      model="gpt-4o-mini",
      messages=[
          {
              "role": "system",
              "type": "text",
              "content": "You are a helpful document analysis and summarization agent.",
          },
          {"role": "user", "content": prompt},
      ],
      temperature=0.3,
  )

  return response.choices[0].message.content


# --- Main Execution ---
if __name__ == "__main__":
  pdf_file_path = "sample_document.pdf"  # Replace with your PDF file path

  if not os.path.exists(pdf_file_path):
    print(f"File not found: {pdf_file_path}")
  else:
    print("Extracting text from PDF...")
    document_content = extract_text_from_pdf(pdf_file_path)

    if document_content.strip():
      print("Generating summary with AI agent...")
      summary = summarize_text_with_agent(document_content)
      print("\n=== AI Summary Result ===\n")
      print(summary)
    else:
      print("No readable text found in the PDF (it might be scanned/image-based).")


Extracting text from PDF...
Total pages in PDF: 1
Generating summary with AI agent...


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [4]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful